<h1 style = "color : #0EE071; text-align : center;"><em>Nata Project</em> - Final Notebook</h1>
<p style = "font-size : 16px; text-align: center;">This notebook runs directly from the first, and will be used to create the model.</p>
<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Machine Learning I</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes, Gustavo Franco & Lucas Casimiro</p>
<br>

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.preprocessing import RobustScaler

import scipy.stats as stats
from scipy.stats import chi2_contingency

from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

from sklearn.linear_model import LassoCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [24]:
learn_data = pd.read_csv('Nata_files/learn.csv')
predict_data = pd.read_csv('Nata_files/predict.csv')

Data Preprocessing

In [25]:
# Initial Data Cleaning
# Turning every string in 'origin' and 'pastry_type' columns to lowercase and stripping whitespace
learn_data['origin'] = learn_data['origin'].astype(str).str.lower().str.strip()
learn_data['pastry_type'] = learn_data['pastry_type'].astype(str).str.lower().str.strip()

# Solving the problem of different namings for the same pastry
learn_data['pastry_type'] = learn_data['pastry_type'].replace('pastel nata', 'pastel de nata')  
learn_data['origin'] = learn_data['origin'].replace({'nan': np.nan})
learn_data['pastry_type'] = learn_data['pastry_type'].replace({'nan': np.nan})

# Dropping Unnecessary or Problematic Rows/Columns
learn_data.dropna(subset=['quality_class'], inplace=True)
learn_data.drop(columns=['pastry_type'], inplace=True)
learn_data.drop('notes_baker', axis=1, inplace=True)
learn_data.drop_duplicates(inplace=True)
learn_data.reset_index(drop=True, inplace=True)


# Handling Impossible Values
# Setting these values to NaN instead of removing the rows, to avoid losing too much data

# Sugar content cannot exceed 75g per 100g
learn_data.loc[learn_data['sugar_content'] > 75, 'sugar_content'] = np.nan

# Fat percentage cannot exceed 100%
learn_data.loc[learn_data['cream_fat_content'] > 100, 'cream_fat_content'] = np.nan

# Salt > 100g per kg is inedible 
learn_data.loc[learn_data['salt_ratio'] > 100, 'salt_ratio'] = np.nan

# Eggs cook at ~65C. 170ºC or 575ºC, for example, is impossible for raw egg addition
learn_data.loc[learn_data['egg_temperature'] > 100, 'egg_temperature'] = np.nan

# Oven/Final temp > 500ºC is likely an error (or a mistake using ºF instead of ºC)
learn_data.loc[learn_data['final_temperature'] > 400, 'final_temperature'] = np.nan
learn_data.loc[learn_data['oven_temperature'] > 400, 'oven_temperature'] = np.nan

# Scaling 
# Define X (features) and y (target)
# Manually mapping the target variable to ensure '1' is the positive class 'OK'
X = learn_data.drop('quality_class', axis=1)
y = learn_data['quality_class'].map({'OK': 1, 'KO': 0}).astype(int)
X_predict = predict_data.copy()

# Split into training and testing sets, using stratify to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Handling Missing Values
# 1. Identify Column Types 
# We treat them differently: Numbers get Median, Text gets Mode (Most Frequent)
numerical_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

# 2. Calculate Statistics (ON TRAIN ONLY) 
# This prevents data leakage. We learn from Train, and apply to Test.
train_medians = X_train[numerical_cols].median()
train_modes = X_train[categorical_cols].mode().iloc[0]

# 3. Impute Missing Values 
# Fill Numerical
X_train[numerical_cols] = X_train[numerical_cols].fillna(train_medians)
X_test[numerical_cols] = X_test[numerical_cols].fillna(train_medians)

# Fill Categorical
X_train[categorical_cols] = X_train[categorical_cols].fillna(train_modes)
X_test[categorical_cols] = X_test[categorical_cols].fillna(train_modes)


# List of columns that are prone to outliers (Continuous variables)
outlier_cols = ['baking_duration', 'cooling_period', 'sugar_content', 
                'salt_ratio', 'egg_temperature', 'final_temperature', 'oven_temperature',
                'preheating_time', 'vanilla_extract']


for col in outlier_cols:
    # 1. Calculate Limits on TRAINING Data Only (Prevent Leakage)
    lower_limit = X_train[col].quantile(0.01) 
    upper_limit = X_train[col].quantile(0.99) 
    
    # 2. Apply to X_train
    # "Clip" is a faster pandas method that does Flooring and Capping in one line
    X_train[col] = X_train[col].clip(lower=lower_limit, upper=upper_limit)
    
    # 3. Apply to X_test (Using Train limits)
    X_test[col] = X_test[col].clip(lower=lower_limit, upper=upper_limit)
 

# One-Hot Encoding for Categorical Variables
# 1. Select categorical columns if you haven't defined this list yet
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# 2. Initialize the encoder
# handle_unknown='ignore': If X_test has a category not seen in X_train, it won't crash (it sets all columns to 0).
# sparse_output=False: Returns a regular array/dataframe instead of a compressed matrix.
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 3. Fit on Train, Transform both
# We fit only on X_train to avoid data leakage
train_encoded = ohe.fit_transform(X_train[categorical_cols])
test_encoded = ohe.transform(X_test[categorical_cols])

# 4. Convert back to DataFrames with readable column names
new_columns = ohe.get_feature_names_out(categorical_cols)

X_train_cat = pd.DataFrame(train_encoded, columns=new_columns, index=X_train.index)
X_test_cat = pd.DataFrame(test_encoded, columns=new_columns, index=X_test.index)

# 5. Join with numerical columns and remove original categorical columns
X_train_final = pd.concat([X_train[numerical_cols], X_train_cat], axis=1)
X_test_final = pd.concat([X_test[numerical_cols], X_test_cat], axis=1)

# 6. Dropping 'origin_lisboa' as it's redundant
X_train_final.drop('origin_lisboa', axis=1, inplace=True)
X_test_final.drop('origin_lisboa', axis=1, inplace=True)

# Check the result
print(f"Original shape: {X_train.shape}")
print(f"Encoded shape:  {X_train_final.shape}")

Original shape: (4159, 15)
Encoded shape:  (4159, 15)


In [26]:
# ==========================================
# 1. INITIAL CLEANING FUNCTION
# (Stateless: Applied identically to both files)
# ==========================================
def clean_initial(df, is_training=True):
    df = df.copy()
    
    # 1. String Standardization
    if 'origin' in df.columns:
        df['origin'] = df['origin'].astype(str).str.lower().str.strip()
        df['origin'] = df['origin'].replace({'nan': np.nan})
    
    if 'pastry_type' in df.columns:
        df['pastry_type'] = df['pastry_type'].astype(str).str.lower().str.strip()
        df['pastry_type'] = df['pastry_type'].replace('pastel nata', 'pastel de nata')
        df['pastry_type'] = df['pastry_type'].replace({'nan': np.nan})

    # 2. Handle Impossible Values (Set to NaN to be imputed later)
    # Note: We do this for predict.csv too, because 575ºC is impossible there too.
    if 'sugar_content' in df.columns:
        df.loc[df['sugar_content'] > 75, 'sugar_content'] = np.nan
    if 'cream_fat_content' in df.columns:
        df.loc[df['cream_fat_content'] > 100, 'cream_fat_content'] = np.nan
    if 'salt_ratio' in df.columns:
        df.loc[df['salt_ratio'] > 100, 'salt_ratio'] = np.nan
    if 'egg_temperature' in df.columns:
        df.loc[df['egg_temperature'] > 100, 'egg_temperature'] = np.nan
    if 'final_temperature' in df.columns:
        df.loc[df['final_temperature'] > 400, 'final_temperature'] = np.nan
    if 'oven_temperature' in df.columns:
        df.loc[df['oven_temperature'] > 400, 'oven_temperature'] = np.nan

    # 3. Drop Columns
    cols_to_drop = ['pastry_type', 'notes_baker']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

    # 4. Training Specifics
    if is_training:
        # Drop rows with no target
        if 'quality_class' in df.columns:
            df.dropna(subset=['quality_class'], inplace=True)
        # Drop duplicates only in training. 
        # WARNING: Never drop duplicates in predict.csv (you lose submission rows)
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)
        
    return df

# ==========================================
# 2. LEARN STATS FUNCTION
# (Calculates Medians, Modes, Quantiles from TRAIN)
# ==========================================
def get_training_stats(df_train):
    stats = {}
    
    # A. Identify Columns
    # Exclude target 'quality_class' from feature calculations
    features = df_train.drop(columns=['quality_class'], errors='ignore')
    
    stats['num_cols'] = features.select_dtypes(include=['float64', 'int64']).columns.tolist()
    stats['cat_cols'] = features.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # B. Imputation Stats (Medians & Modes)
    stats['medians'] = df_train[stats['num_cols']].median()
    stats['modes'] = df_train[stats['cat_cols']].mode().iloc[0]
    
    # C. Outlier Clipping Bounds (1st and 99th percentile)
    outlier_cols = ['baking_duration', 'cooling_period', 'sugar_content', 
                    'salt_ratio', 'egg_temperature', 'final_temperature', 
                    'oven_temperature', 'preheating_time', 'vanilla_extract']
    
    # Filter to only existing columns
    stats['clip_bounds'] = {}
    for col in outlier_cols:
        if col in df_train.columns:
            stats['clip_bounds'][col] = {
                'lower': df_train[col].quantile(0.01),
                'upper': df_train[col].quantile(0.99)
            }
            
    # D. One-Hot Encoder (Fit on Train)
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    ohe.fit(df_train[stats['cat_cols']])
    stats['ohe'] = ohe
    stats['ohe_cols'] = ohe.get_feature_names_out(stats['cat_cols'])
    
    return stats

# ==========================================
# 3. APPLY TRANSFORMATION FUNCTION
# (Applies the learned stats to ANY dataframe)
# ==========================================
def process_data(df, stats):
    df_proc = df.copy()
    
    # A. Impute Missing Values (Using TRAIN values)
    df_proc[stats['num_cols']] = df_proc[stats['num_cols']].fillna(stats['medians'])
    df_proc[stats['cat_cols']] = df_proc[stats['cat_cols']].fillna(stats['modes'])
    
    # B. Outlier Clipping (Using TRAIN bounds)
    for col, bounds in stats['clip_bounds'].items():
        if col in df_proc.columns:
            df_proc[col] = df_proc[col].clip(lower=bounds['lower'], upper=bounds['upper'])
            
    # C. One-Hot Encoding (Using TRAIN encoder)
    encoded_array = stats['ohe'].transform(df_proc[stats['cat_cols']])
    encoded_df = pd.DataFrame(encoded_array, columns=stats['ohe_cols'], index=df_proc.index)
    
    # D. Merge & Clean
    # Concatenate numerical + encoded
    df_final = pd.concat([df_proc[stats['num_cols']], encoded_df], axis=1)
    
    # E. Drop 'origin_lisboa' (Redundant)
    if 'origin_lisboa' in df_final.columns:
        df_final.drop('origin_lisboa', axis=1, inplace=True)
        
    return df_final

# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================

# Step 1: Initial Cleaning
learn_data = clean_initial(learn_data, is_training=True)
predict_data = clean_initial(predict_data, is_training=False)

# Step 2: Learn Stats from Training Data (Full learn.csv)
training_stats = get_training_stats(learn_data)

# Step 3: Apply Stats to Process Features
X = process_data(learn_data, training_stats)
X_predict_csv = process_data(predict_data, training_stats)
# Step 4: Get Target
y = learn_data['quality_class'].map({'OK': 1, 'KO': 0}).astype(int)

# Double Check Dimensions
print(f"Training Shape: {X.shape}")
print(f"Prediction Shape: {X_predict_csv.shape}")

# Safety check: Ensure columns match exactly
X_predict_csv = X_predict_csv[X.columns]

Training Shape: (5199, 16)
Prediction Shape: (1300, 16)


Feature Selection

In [27]:
vars_to_drop = ['ambient_humidity', 'cream_fat_content', 'lemon_zest_ph', 'oven_temperature']

# Drop from Training Data
X = X.drop(columns=[c for c in vars_to_drop if c in X.columns])

# Drop from Prediction Data (CRITICAL!)
X_predict_csv = X_predict_csv.drop(columns=[c for c in vars_to_drop if c in X_predict_csv.columns])

Modeling

In [ ]:
# 1. Define the Base Learners with your specific tuned parameters
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_split=10,
    max_depth=20,
    random_state=42
)

# Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8, #we use a subsample to limit overfitting
    random_state=42
)

# AdaBoost (Note the nested base estimator for max_depth)
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=5),
    n_estimators=150,
    learning_rate=0.1,
    random_state=42
)

# 2. Define the Final Estimator (Meta-learner)
meta_learner = LogisticRegression(
    C=0.1,
    solver='lbfgs',
    random_state=42
)

# 3. Construct the Stacking Classifier
# The names 'rf', 'gb', 'adabosting' must match the prefixes used in your grid search
final_model = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('gb', gb_model),
        ('adabosting', ada_model)
    ],
    final_estimator=meta_learner,
    cv=5) # Optional: Internal CV for stacking

In [29]:
final_model.fit(X, y)

,estimators,"[('rf', ...), ('gb', ...), ...]"
,final_estimator,LogisticRegre...ndom_state=42)
,cv,5
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,n_estimators,300
,criterion,'gini'
,max_depth,20
,min_samples_split,10


In [30]:
y_pred_numeric = final_model.predict(X_predict_csv)

# 2. Convert Numeric Predictions back to Labels
# Since we mapped {'OK': 1, 'KO': 0}, we must reverse it.
# 1 -> 'OK'
# 0 -> 'KO'
y_pred_labels = ['OK' if pred == 1 else 'KO' for pred in y_pred_numeric]

# 3. Create the Submission DataFrame
submission = pd.DataFrame()

# CRITICAL: We need to grab the ID column from the raw data.
# If 'predict.csv' has a column named 'row_id' or 'id', we use it.
if 'row_id' in predict_data.columns:
    submission['row_id'] = predict_data['row_id']
elif 'id' in predict_data.columns:
    submission['id'] = predict_data['id']
else:
    # Fallback: If no ID column exists, we use the dataframe index
    submission['row_id'] = predict_data.index

# Add your predictions
submission['quality_class'] = y_pred_labels

# 4. Save to CSV
# Replace 'ML08' with your actual group number
submission.to_csv('ML08_Submission.csv', index=False)

# 5. Verify the Output
print("Submission file created successfully!")
print(submission.head())
print(f"Total rows: {len(submission)}")


Submission file created successfully!
     id quality_class
0  5201            KO
1  5202            KO
2  5203            OK
3  5204            OK
4  5205            OK
Total rows: 1300
